# Big Data Bowl 2027 — Predicting Injury from the Combine, and How Reliable Its Movement Measures Are

**What this notebook does:** estimates how well current Combine data can predict rookie injury once playing time is properly accounted for, by modeling games missed per snap played and testing whether tracked movement adds anything beyond position, draft round, and timed results. It also measures how reliable each movement trait already is when averaged across all the drills a player ran, and uses that reliability to estimate how strong each trait's link to injury would be with a noise-free measurement, the target the proposed drill is built to reach.

**Sections**
1. Rookie injury and exposure (snaps played)
2. Combine movement traits (per drill, then averaged across drills)
3. Reliability: within a drill (attempt 1 vs. 2) and across drills, for all players and **DBs only**
4. Exposure-adjusted injury model: games missed per snap (rate ratios), plus a cross-validated comparison
5. Correcting for measurement noise: observed vs. noise-free association, and the sample needed to detect it

**Pre-specified features** (fixed before running): weight and 40 time (timed); speed kept through turns, braking, and turning (tracked). No untouched holdout remains (the 2025 class was used for confirmation), so the model results here are descriptive; the reliability results don't involve injury data.

**Data:** reads `../data/` and `external/` (run from `Ben_EDA/`); reuses `outputs/rookie_injury/combine_attempts.parquet` if the rookie-injury notebook saved it. Without `../data/`, it runs on synthetic data from `synthetic_rookie.py`.

## 0. Setup

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid")

DATA, EXT = Path("../data"), Path("external")
CACHE = Path("outputs/rookie_injury/combine_attempts.parquet")
OUT_DIR = Path("outputs/injury_prediction")
OUT_DIR.mkdir(parents=True, exist_ok=True)
USE_SYNTHETIC = not (DATA / "players.csv").exists()

SEASONS = [2023, 2024, 2025]
IR_MIN_GAMES = 3
EXCLUDE_INJURY = "concussion|head|illness|personal|rest|not injury|non-football"
MIN_GAMES = 4
FAMILY = {"CB": "DB", "FS": "DB", "SS": "DB", "S": "DB", "DB": "DB", "WR": "Receivers", "TE": "Receivers",
          "T": "OL", "G": "OL", "C": "OL", "DE": "Front seven", "DT": "Front seven", "NT": "Front seven",
          "OLB": "Front seven", "ILB": "Front seven", "MLB": "Front seven", "LB": "Front seven"}
TIMED = ["combine_weight", "forty"]
TRAITS = {"speed_kept": "retention", "braking": "max_decel", "turning": "max_turn", "first_step": "first_step"}
TRACKED = ["speed_kept", "braking", "turning"]          # pre-specified model features (first_step: reliability only)
MIN_SPEED_FOR_DIR, HARD_CUT_DEG_S, RETENTION_SEC, FIRST_STEP_SEC, ACC_CLIP = 2.0, 150, 0.5, 0.8, 12.0
rng = np.random.default_rng(0)

if USE_SYNTHETIC:
    from synthetic_rookie import make_all
    R = make_all()
    players, combine_res, combine_trk = R["players"], R["combine_results"], R["combine_tracking"]
    ext_players, injuries, snaps, schedule = R["ext_players"], R["injuries"], R["snaps"], R["schedule"]
else:
    players = pd.read_csv(DATA / "players.csv")
    combine_res = pd.read_csv(DATA / "combine_results.csv")
    combine_trk = None if CACHE.exists() else pd.read_csv(DATA / "combine_tracking.csv")
    ext_players = pd.read_csv(EXT / "players.csv", usecols=["gsis_id", "nfl_id", "pfr_id", "display_name", "position"])
    injuries = pd.concat(pd.read_csv(EXT / f"injuries_{y}.csv") for y in SEASONS)
    import nflreadpy as nfl
    snaps = nfl.load_snap_counts(SEASONS).to_pandas()
    schedule = nfl.load_schedules(SEASONS).to_pandas()

idmap = ext_players.dropna(subset=["nfl_id"]).copy()
idmap["nfl_id"] = idmap.nfl_id.astype(int)
players["family"] = players.nfl_position.map(FAMILY).fillna("Other")
players["draft_round"] = pd.to_numeric(players.get("draft_round"), errors="coerce").fillna(8)
print("Synthetic demo mode" if USE_SYNTHETIC else "Real data")
print(players.family.value_counts().to_string())

Real data
family
Receivers      148
Front seven    121
OL             121
DB             118
Other            2


## 1. Rookie injury and exposure

Games missed with an injury in the draft-year season (weeks listed Out, plus 3+ game absences right after an injury listing, which catches injured reserve; concussions and illness excluded), alongside **snaps played**, the exposure. A player with no snaps had no chance to be injured in a game, so rate models use only players who played.

In [2]:
def rookie_outcome(players, injuries, snaps, schedule, idmap):
    inj = injuries[injuries.game_type == "REG"].merge(idmap[["gsis_id", "nfl_id"]], on="gsis_id")
    inj = inj.merge(players[["nfl_id", "draft_year"]], on="nfl_id")
    inj = inj[(inj.season == inj.draft_year) & inj.report_primary_injury.notna()
              & ~inj.report_primary_injury.astype(str).str.lower().str.contains(EXCLUDE_INJURY)]
    out_weeks = inj[inj.report_status == "Out"].groupby("nfl_id").week.apply(set).to_dict()
    listed = inj.groupby("nfl_id").week.apply(set).to_dict()
    sn = snaps[snaps.game_type == "REG"].merge(idmap[["pfr_id", "nfl_id"]], left_on="pfr_player_id", right_on="pfr_id")
    sn = sn.merge(players[["nfl_id", "draft_year"]], on="nfl_id")
    sn = sn[sn.season == sn.draft_year]
    sch = schedule[schedule.game_type == "REG"]
    team_weeks = (pd.concat([sch[["season", "week", "home_team"]].rename(columns={"home_team": "team"}),
                             sch[["season", "week", "away_team"]].rename(columns={"away_team": "team"})])
                    .groupby(["season", "team"]).week.apply(lambda w: sorted(set(w))).to_dict())
    by_player = {k: g for k, g in sn.groupby("nfl_id")}
    rows = []
    for nid in players.nfl_id:
        weeks, ir = set(out_weeks.get(nid, set())), set()
        g = by_player.get(nid)
        if g is not None and nid in listed:
            played, run = set(g.week), []
            for w in team_weeks.get((g.season.iloc[0], g.team.mode().iloc[0]), []) + [None]:
                if w is not None and w not in played and w > min(played):
                    run.append(w); continue
                if len(run) >= IR_MIN_GAMES and any(run[0] - 2 <= lw <= run[0] for lw in listed[nid]):
                    ir |= set(run)
                run = []
        rows.append(dict(nfl_id=nid, games_out=len(weeks | ir), games_played=0 if g is None else len(g),
                         snaps=0 if g is None else int((g.offense_snaps + g.defense_snaps).sum())))
    return pd.DataFrame(rows)


P = players.merge(rookie_outcome(players, injuries, snaps, schedule, idmap), on="nfl_id")
P["injured"] = P.games_out > 0
P = P[P.family != "Other"]
print(f"{len(P)} players | played at least one snap: {(P.snaps > 0).sum()} | injured: {P.injured.sum()}")
print(P.groupby("family").agg(players=("nfl_id", "size"), injured=("injured", "sum"),
                              median_snaps=("snaps", "median")).to_string())

508 players | played at least one snap: 380 | injured: 101
             players  injured  median_snaps
family                                     
DB               118       24         149.5
Front seven      121       18         132.0
OL               121       21          89.0
Receivers        148       38         156.0


## 2. Combine movement traits

Per drill attempt: speed kept through turns, braking (peak deceleration), turning (peak turn rate), and first step. Each is z-scored **within drill**, so drills are comparable. A player's trait is the average over all the drills he ran (first attempts), the cross-drill composite.

In [3]:
COMBINE_KEYS = ["nfl_id", "event_id", "drill_name", "attempt"]


def combine_attempt_features(ct):
    ct = ct.copy()
    ct["time"] = pd.to_datetime(ct.time, format="mixed")
    ct = ct.sort_values(COMBINE_KEYS + ["time"])
    fps = float(round(1 / ct.groupby(COMBINE_KEYS).time.diff().dt.total_seconds().median()))
    ct["start_t"] = ct.time.where(ct.s >= 0.3).groupby([ct[k] for k in COMBINE_KEYS]).transform("min")
    ct["t"] = (ct.time - ct.start_t).dt.total_seconds()
    ct = ct[ct.t >= 0].copy()
    g = ct.groupby(COMBINE_KEYS, sort=False)
    x0, y0 = g.x.transform("first"), g.y.transform("first")
    ct["disp"] = np.hypot(ct.x - x0, ct.y - y0)
    ct["long_acc"] = g.s.diff() * fps
    ct["long_acc"] = ct.groupby(COMBINE_KEYS).long_acc.transform(
        lambda v: v.rolling(3, center=True, min_periods=1).mean()).clip(-ACC_CLIP, ACC_CLIP)
    unwrapped = g.dir.transform(lambda d: np.rad2deg(np.unwrap(np.deg2rad(d.to_numpy()))))
    ct["turn"] = unwrapped.groupby([ct[k] for k in COMBINE_KEYS]).diff().abs() * fps
    ct.loc[ct.s < MIN_SPEED_FOR_DIR, "turn"] = np.nan
    w = max(1, int(round(RETENTION_SEC * fps)))
    is_cut = (ct.turn >= HARD_CUT_DEG_S) & (ct.turn == ct.groupby(COMBINE_KEYS).turn.transform(
        lambda v: v.rolling(5, center=True, min_periods=1).max()))
    s_b, s_a = g.s.shift(w), g.s.shift(-w)
    ct["retention"] = np.where(is_cut & (s_b > MIN_SPEED_FOR_DIR), s_a / s_b, np.nan)
    ct["fs"] = np.where(ct.t <= FIRST_STEP_SEC, ct.long_acc, np.nan)
    ct["t1"] = np.where(ct.disp >= 1, ct.t, np.nan)
    out = (ct.groupby(COMBINE_KEYS)
             .agg(first_step=("fs", "max"), time_1yd=("t1", "min"), max_decel=("long_acc", "min"),
                  max_turn=("turn", "max"), retention=("retention", "mean"))
             .reset_index())
    out["max_decel"] = -out.max_decel
    return out


if CACHE.exists() and not USE_SYNTHETIC:
    att = pd.read_parquet(CACHE)
else:
    ct = combine_trk.dropna(subset=["nfl_id"]).copy()
    ct["drill_name"] = ct.drill_name.astype(str).str.upper().str.replace("-", "_", regex=False)
    att = combine_attempt_features(ct)
MEAS = list(TRAITS.values())
z = att.copy()
z[MEAS] = z.groupby("drill_name")[MEAS].transform(lambda x: (x - x.mean()) / x.std())
first = z[z.attempt == z.groupby(["nfl_id", "drill_name"]).attempt.transform("min")]
comp = first.groupby("nfl_id")[MEAS].mean().rename(columns={v: k for k, v in TRAITS.items()})
n_drills = first.groupby("nfl_id")[MEAS].count().rename(columns={v: f"n_drills_{k}" for k, v in TRAITS.items()})
P = P.merge(comp.join(n_drills).reset_index(), on="nfl_id", how="left")
P = P.merge(combine_res.drop_duplicates("nfl_id", keep="last")[["nfl_id"] + TIMED], on="nfl_id", how="left")
FEATS = TIMED + list(TRAITS)
P[FEATS] = P.groupby("family")[FEATS].transform(lambda x: (x - x.mean()) / x.std())
print("drills per player with each trait (median):")
print(P[[c for c in P.columns if c.startswith("n_drills_")]].median().to_string())

drills per player with each trait (median):
n_drills_speed_kept    2.0
n_drills_braking       9.0
n_drills_turning       9.0
n_drills_first_step    9.0


## 3. Reliability

**Within a drill:** attempt 1 vs. attempt 2 of the same drill (the result behind the reps argument), with 95% bootstrap intervals, for all players and DBs only.

**Across drills:** each player's drills are randomly split into two halves and the trait averaged within each half. How well the two halves agree, stepped up with the Spearman-Brown formula, is the reliability of the cross-drill composite used as a feature. Repeated over 50 random splits.

In [4]:
def sb_reps(r, target=0.8):
    return target * (1 - r) / (r * (1 - target)) if r > 0 else np.inf


def boot_ci(f, data, n_boot=1000):
    vals = [f(data.iloc[rng.integers(0, len(data), len(data))]) for _ in range(n_boot)]
    return np.nanpercentile(vals, [2.5, 97.5])


def within_drill(ids, label):
    rows = []
    zz = att[att.attempt.isin([1, 2]) & att.nfl_id.isin(ids)].copy()
    zz[MEAS] = zz.groupby("drill_name")[MEAS].transform(lambda x: (x - x.mean()) / x.std())
    for trait, m in TRAITS.items():
        w = zz.pivot_table(index=["nfl_id", "drill_name"], columns="attempt", values=m).dropna()
        if not {1, 2} <= set(w.columns) or len(w) < 10:
            rows.append(dict(group=label, trait=trait, pairs=len(w)))
            continue
        f = lambda d: d[1].corr(d[2], method="spearman")
        r = f(w); lo, hi = boot_ci(f, w)
        rows.append(dict(group=label, trait=trait, pairs=len(w), r=r, r_lo=lo, r_hi=hi,
                         reps_for_0_8=sb_reps(r), reps_lo=sb_reps(hi), reps_hi=sb_reps(lo)))
    return rows


def split_half(ids, trait, n_splits=50):
    m = TRAITS[trait]
    d = first[first.nfl_id.isin(ids)][["nfl_id", "drill_name", m]].dropna()
    d = d[d.groupby("nfl_id")[m].transform("size") >= 2]
    if d.nfl_id.nunique() < 15:
        return dict(players=d.nfl_id.nunique())
    rs = []
    for s in range(n_splits):
        r_ = np.random.default_rng(s)
        d = d.assign(half=d.groupby("nfl_id")[m].transform(lambda x: r_.permutation(len(x)) % 2))
        h = d.groupby(["nfl_id", "half"])[m].mean().unstack().dropna()
        rs.append(h[0].corr(h[1], method="spearman"))
    r_half = float(np.mean(rs))
    return dict(players=d.nfl_id.nunique(), median_drills=float(d.groupby("nfl_id").size().median()),
                split_half_r=r_half, composite_reliability=2 * r_half / (1 + r_half) if r_half > 0 else np.nan)


all_ids, db_ids = set(P.nfl_id), set(P.loc[P.family == "DB", "nfl_id"])
WITHIN = pd.DataFrame(within_drill(all_ids, "All") + within_drill(db_ids, "DB"))
ACROSS = pd.DataFrame([dict(group=g, trait=t) | split_half(ids, t) for g, ids in [("All", all_ids), ("DB", db_ids)]
                       for t in TRAITS])
print("Within a drill: attempt 1 vs. attempt 2")
display(WITHIN.round(2))
print("Across drills: reliability of the cross-drill composite")
display(ACROSS.round(2))
WITHIN.to_csv(OUT_DIR / "reliability_within_drill.csv", index=False)
ACROSS.to_csv(OUT_DIR / "reliability_across_drills.csv", index=False)

Within a drill: attempt 1 vs. attempt 2


,group,trait,pairs,r,r_lo,r_hi,reps_for_0_8,reps_lo,reps_hi
0,All,speed_kept,11,0.30,-0.38,0.83,9.33,0.84,inf
1,All,braking,891,0.28,0.21,0.34,10.27,7.68,14.79
2,All,turning,890,0.30,0.23,0.36,9.50,7.08,13.42
3,All,first_step,891,0.32,0.26,0.39,8.50,6.30,11.63
4,DB,speed_kept,1,NaN,NaN,NaN,NaN,NaN,NaN
5,DB,braking,244,0.39,0.26,0.50,6.38,4.03,11.35
6,DB,turning,244,0.42,0.31,0.53,5.51,3.54,9.08
7,DB,first_step,244,0.28,0.15,0.40,10.29,5.99,22.02


Across drills: reliability of the cross-drill composite


,group,trait,players,median_drills,split_half_r,composite_reliability
0,All,speed_kept,260,3.0,0.07,0.13
1,All,braking,508,9.0,0.43,0.60
2,All,turning,508,9.0,0.39,0.56
3,All,first_step,508,9.0,0.43,0.61
4,DB,speed_kept,88,2.0,-0.03,NaN
5,DB,braking,118,9.0,0.11,0.19
6,DB,turning,118,9.0,-0.11,NaN
7,DB,first_step,118,9.0,0.41,0.58


## 4. Exposure-adjusted injury model

**Rate model:** games missed modeled as a rate per snap played (Poisson regression with snaps as the exposure, robust standard errors), for players with at least one snap. Coefficients are **rate ratios per standard deviation** of each feature, within position family: below 1 means fewer games missed per snap. Fitted for all families (with family and draft round) and for DBs alone.

**Prediction check:** cross-validated AUC for "injured or not" among players with 4+ games, comparing a base model (family, draft round, log snaps) with timed results added and with tracked traits added. Descriptive only, since no untouched holdout remains.

In [5]:
def rate_model(df, feats, label):
    d = df[df.snaps > 0].copy()
    X = pd.DataFrame(index=d.index)
    if d.family.nunique() > 1:
        X = pd.get_dummies(d.family, prefix="fam", drop_first=True).astype(float)
    X["draft_round"] = d.draft_round
    for f in feats:
        X[f] = d[f].fillna(0.0)                      # z-scores: missing -> family average
        X[f"{f}_missing"] = d[f].isna().astype(float)
    X = X.loc[:, X.std() > 0]
    X = sm.add_constant(X)
    fit = sm.GLM(d.games_out, X, family=sm.families.Poisson(), offset=np.log(d.snaps)).fit(cov_type="HC0")
    ci = fit.conf_int()
    rows = [dict(model=label, feature=f, rate_ratio_per_sd=np.exp(fit.params[f]), ci_low=np.exp(ci.loc[f, 0]),
                 ci_high=np.exp(ci.loc[f, 1]), p_value=fit.pvalues[f]) for f in feats if f in fit.params]
    return pd.DataFrame(rows), fit, len(d), int(d.injured.sum())


RATE = []
for label, df in [("All families", P), ("DBs only", P[P.family == "DB"])]:
    tab, fit, n, k = rate_model(df, TIMED + TRACKED, label)
    print(f"{label}: {n} players with snaps, {k} injured")
    RATE.append(tab)
RATE = pd.concat(RATE, ignore_index=True)
display(RATE.round(3))
RATE.to_csv(OUT_DIR / "rate_ratios.csv", index=False)

All families: 380 players with snaps, 92 injured
DBs only: 85 players with snaps, 23 injured


,model,feature,rate_ratio_per_sd,ci_low,ci_high,p_value
0,All families,combine_weight,0.896,0.645,1.243,0.510
1,All families,forty,1.139,0.845,1.535,0.393
2,All families,speed_kept,0.832,0.626,1.106,0.205
3,All families,braking,1.305,0.923,1.845,0.131
4,All families,turning,0.650,0.446,0.947,0.025
5,DBs only,combine_weight,0.761,0.385,1.503,0.431
6,DBs only,forty,1.135,0.516,2.496,0.753
7,DBs only,speed_kept,1.048,0.691,1.588,0.826
8,DBs only,braking,1.725,0.723,4.115,0.219
9,DBs only,turning,0.451,0.237,0.856,0.015


In [6]:
def model_matrix(df, cols):
    X = pd.get_dummies(df.family, prefix="fam", drop_first=True).astype(float)
    X["draft_round"] = df.draft_round.values
    X["log_snaps"] = np.log1p(df.snaps.values)
    for c in cols:
        X[c] = df[c].values
    return X


def cv_auc(df, sets, n_repeats=20):
    y = df.injured.astype(int).to_numpy()
    cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=n_repeats, random_state=0)
    scores = {k: [] for k in sets}
    for tr, te in cv.split(np.zeros(len(y)), y):
        for k, cols in sets.items():
            X = model_matrix(df, cols).to_numpy()
            m = make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler(),
                              LogisticRegression(C=0.5, max_iter=5000)).fit(X[tr], y[tr])
            scores[k].append(roc_auc_score(y[te], m.predict_proba(X[te])[:, 1]))
    s = pd.DataFrame(scores)
    out = s.agg(["mean", "std"]).T
    out["vs_base"] = [(s[c] - s["base"]).mean() for c in out.index]
    out["share_beating_base"] = [((s[c] - s["base"]) > 0).mean() for c in out.index]
    return out


EXP = P[P.games_played >= MIN_GAMES]
AUC = cv_auc(EXP, {"base": [], "base + timed": TIMED, "base + timed + tracked": TIMED + TRACKED,
                   "base + tracked": TRACKED})
print(f"Cross-validated AUC, {len(EXP)} players with {MIN_GAMES}+ games ({EXP.injured.sum()} injured)")
display(AUC.round(3))
AUC.to_csv(OUT_DIR / "cv_auc.csv")

Cross-validated AUC, 353 players with 4+ games (89 injured)


,mean,std,vs_base,share_beating_base
base,0.536,0.068,0.000,0.00
base + timed,0.559,0.062,0.023,0.67
base + timed + tracked,0.554,0.065,0.018,0.63
base + tracked,0.542,0.070,0.006,0.59


## 5. Correcting for measurement noise

A noisy measure shrinks any real relationship. With the reliability of each cross-drill composite from Section 3, the **Spearman correction** estimates the association with a noise-free measure: corrected r = observed r ÷ √reliability. From that, the number of players needed to detect it (80% power, two-sided 0.05) is shown at today's reliability and at the drill's target of 0.8 (a trait already above 0.8 keeps its current value). A blank means the association is too close to zero to detect at any realistic sample size. If a composite's split-half agreement is zero or negative, its reliability can't be estimated and the correction is left blank.

The observed association is the rank correlation between the trait and being injured, among players with 4+ games, adjusted for family and log snaps by correlating residuals. Corrections of small, noisy correlations are themselves noisy, so the bootstrap intervals matter as much as the point values.

In [7]:
def residualize(df, col):
    X = sm.add_constant(pd.concat([pd.get_dummies(df.family, drop_first=True).astype(float),
                                   np.log1p(df.snaps).rename("log_snaps")], axis=1))
    return sm.OLS(df[col].astype(float), X, missing="drop").fit().resid


def adjusted_r(df, trait):
    d = df.dropna(subset=[trait])
    if len(d) < 20 or d.injured.nunique() < 2:
        return np.nan
    return stats.spearmanr(residualize(d, trait), residualize(d, "injured")).correlation


def n_needed(r):
    """Players needed to detect r (80% power, two-sided 0.05); blank when r is too close to 0 to matter."""
    return int(np.ceil(((1.96 + 0.84) / np.arctanh(abs(r))) ** 2 + 3)) if np.isfinite(r) and abs(r) >= 0.03 else np.nan


rows = []
for label, df, fam in [("All families", EXP, "All"), ("DBs only", EXP[EXP.family == "DB"], "DB")]:
    for trait in TRACKED:
        rel = ACROSS[(ACROSS.group == fam) & (ACROSS.trait == trait)].composite_reliability
        rel = float(rel.iloc[0]) if len(rel) and pd.notna(rel.iloc[0]) else np.nan
        r_obs = adjusted_r(df, trait)
        lo, hi = boot_ci(lambda d: adjusted_r(d, trait), df.dropna(subset=[trait]), n_boot=500)
        r_true = r_obs / np.sqrt(rel) if rel and rel > 0 else np.nan
        target = max(rel, 0.8) if np.isfinite(rel) else 0.8           # a trait already above 0.8 stays where it is
        r_target = r_true * np.sqrt(target) if np.isfinite(r_true) else np.nan
        rows.append(dict(group=label, trait=trait, n=int(df[trait].notna().sum()), reliability_now=rel,
                         r_observed=r_obs, r_obs_lo=lo, r_obs_hi=hi, r_noise_free=r_true,
                         reliability_target=target, r_at_target=r_target,
                         players_needed_now=n_needed(r_obs), players_needed_at_target=n_needed(r_target)))
ATT = pd.DataFrame(rows)
display(ATT.round(3))
ATT.to_csv(OUT_DIR / "attenuation_correction.csv", index=False)

,group,trait,n,reliability_now,r_observed,r_obs_lo,r_obs_hi,r_noise_free,reliability_target,r_at_target,players_needed_now,players_needed_at_target
0,All families,speed_kept,274,0.126,-0.126,-0.231,-0.014,-0.355,0.8,-0.317,491.0,76.0
1,All families,braking,353,0.600,-0.011,-0.100,0.080,-0.014,0.8,-0.012,NaN,NaN
2,All families,turning,353,0.560,-0.011,-0.115,0.071,-0.015,0.8,-0.014,NaN,NaN
3,DBs only,speed_kept,81,NaN,-0.108,-0.295,0.070,NaN,0.8,NaN,665.0,NaN
4,DBs only,braking,83,0.194,-0.004,-0.195,0.169,-0.009,0.8,-0.008,NaN,NaN
5,DBs only,turning,83,NaN,-0.085,-0.273,0.180,NaN,0.8,NaN,1075.0,NaN


### Summary

Read in this order:
1. **Section 3, DB rows:** do DBs show the same low attempt-to-attempt reliability as all players? This closes the DB-only gap in the pitch.
2. **Section 3, across drills:** how far does averaging over the drills a player already runs get each trait? If it stays well below 0.8, current drills can't fix the problem by themselves.
3. **Section 4:** do tracked traits change injury rate per snap, or improve prediction, beyond position, draft round, and timed results? Expect modest or no gains; that's the current ceiling.
4. **Section 5:** the noise-free estimate and the sample needed at today's reliability vs. at 0.8. The gap between those two numbers is the drill's value for answering the injury question.